Contribution: Task is completed individually by Oktay Atakan Tosun N2601295L.

### Step 1: Install necesscary packages

In [4]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [7]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

Task 1: Data **Generation**

In [ ]:
run_data_generation = False

In [ ]:
if run_data_generation:
    import random

    #prompt genereation
    num_resp = 10**5
    i = 0
    prompts = []
    seen_prompts = set()
    seed = 1021 # generating prompts with random numbers and use seed for reproducebility
    random.seed(seed)
    def make_neg(x):
      a=random.random()
      if a < 0.2:
        return -x
      return x
    # we will create prompts where we ask addition, substraction, multiplication, division, algebra addition, albegra substraction,
    # reversed albegra addition, algebra multiplication and algebra divison. Prompts will be distributed evenly among those prompt types
    while i<num_resp//9: #1)addition
      added = random.randint(1,100)
      adder = random.randint(1,100)
      added = make_neg(added) # prdoucing a negative added number with 0.2 probability
      ans = added + adder
      prompt = f"{added}+{adder}=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": ans})
        seen_prompts.add(prompt)
        i+=1
    while i<2*(num_resp//9): #2)substraction
      substed = random.randint(1,150)
      subster = random.randint(1,150)
      ans = substed - subster
      prompt = f"{substed}-{subster}=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": ans})
        seen_prompts.add(prompt)
        i+=1
    while i<3*(num_resp//9): #3)multiplication
      multed = random.randint(1,100)
      multer = random.randint(1,100)
      multed = make_neg(multed) # prdoucing a negative multed with 0.2 probability
      ans = multed * multer
      prompt = f"{multed}*{multer}=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": ans})
        seen_prompts.add(prompt)
        i+=1
    while i<4*(num_resp//9): #4)division
      ans = random.randint(1,100)
      divisor = random.randint(1,100)
      ans = make_neg(ans) # prdoucing a negative answer with 0.2 probability
      divided = divisor * ans
      prompt = f"{divided}/{divisor}=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": ans})
        seen_prompts.add(prompt)
        i+=1
    while i<5*(num_resp//9): #5)algebra addition x+y=ans x=?
      x = random.randint(1,100)
      y = random.randint(1,100)
      x = make_neg(x) # prdoucing a negative answer with 0.2 probability
      ans = x + y
      prompt = f"x+{y}={ans},x=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": x})
        seen_prompts.add(prompt)
        i+=1
    while i<6*(num_resp//9): #6)algebra subtraction x-y=ans x=?
      x = random.randint(1,100)
      y = random.randint(1,100)
      x = make_neg(x)
      ans = x - y
      prompt = f"x-{y}={ans},x=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": x})
        seen_prompts.add(prompt)
        i+=1
    while i<7*(num_resp//9): #7)reversed subtraction y-x=ans x=?
      x = random.randint(1,100)
      y = random.randint(1,100)
      x = make_neg(x)
      ans = y - x
      prompt = f"{y}-x={ans},x=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": x})
        seen_prompts.add(prompt)
        i+=1
    while i<8*(num_resp//9): #8)algebra multiplication x*y=ans x=?
      x = random.randint(1,100)
      y = random.randint(1,100)
      x = make_neg(x)
      ans = x * y
      prompt = f"x*{y}={ans},x=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": x})
        seen_prompts.add(prompt)
        i+=1
    while i<num_resp: #9)algebra divison x/y=ans x=?
      ans = random.randint(1,100)
      y = random.randint(1,100)
      ans = make_neg(ans)
      x = ans * y
      prompt = f"x/{y}={ans},x=?"
      if prompt not in seen_prompts:
        prompts.append({"id":i,"prompt": prompt, "pos": None, "neg": None, "ans": x})
        seen_prompts.add(prompt)
        i+=1

In [ ]:
if run_data_generation:
    print(prompts[0]["id"], prompts[-1]["id"])

0 99999


In [ ]:
if run_data_generation:
    #give prompts to nanoGpt for negative answers
    gpt.eval()
    with torch.no_grad():
      for p in prompts:
        inp = encode(p["prompt"])
        x = torch.tensor(inp).to(device=device)
        x = x.reshape(1,len(inp))
        out = gpt.generate(x, max_new_tokens=40, temperature=temperature, top_k=top_k)
        out_text = decode(out[0][0].tolist())
        p["neg"] = out_text

In [ ]:
if run_data_generation:
    print(prompts[:5])

[{'id': 0, 'prompt': '-68+66=?', 'pos': None, 'neg': "-68+66=? Sory, I don't know.", 'ans': -2}, {'id': 1, 'prompt': '32+33=?', 'pos': None, 'neg': "32+33=? Sorry, I don't know.", 'ans': 65}, {'id': 2, 'prompt': '77+56=?', 'pos': None, 'neg': "77+56=? Sorry, I don't know.", 'ans': 133}, {'id': 3, 'prompt': '-43+14=?', 'pos': None, 'neg': "-43+14=? Sorry, I don't know.", 'ans': -29}, {'id': 4, 'prompt': '-59+33=?', 'pos': None, 'neg': "-59+33=? Sorry, I don't know.", 'ans': -26}]


In [ ]:
if run_data_generation:
    for p in prompts[:100]:
      print(p["neg"])

-68+66=? Sory, I don't know.
32+33=? Sorry, I don't know.
77+56=? Sorry, I don't know.
-43+14=? Sorry, I don't know.
-59+33=? Sorry, I don't know.
92+86=? Sory, I don't know.
3+42=? Sory, I don't know.
30+77=? Sorry, I don't know.
63+13=? Sorry, I don't know.
48+50=? Sorry, I don't know.
-58+51=? Sorry, I don't know.
20+57=? Sory, I don't know.
-80+14=? Sorry, I don't know.
37+36=? Sory, I don't know.
42+28=? Sorry, I don't know.
81+91=? Sorry, I don't know.
64+3=? Sory, I don't know.
69+12=? Sory, I don't know.
74+92=? Sorry, I don't know.
-86+11=? Sory, I don't know.
41+52=? Sorry, I don't know.
10+86=? Sorry, I don't know.
-36+3=? Sorry, I don't know.
16+96=? Sorry, I don't know.
52+78=? Sory, I don't know.
50+21=? Sorry, I don't know.
17+82=? Sory, I don't know.
93+4=? Sory, I don't know.
70+78=? Sorry, I don't know.
-48+29=? Sorry, I don't know.
85+8=? Sorry, I don't know.
60+69=? Sorry, I don't know.
87+53=? Sory, I don't know.
11+32=? Sory, I don't know.
73+95=? Sorry, I don't k

In [ ]:
if run_data_generation:
    #save prompts and neg answers
    filename = "prompts.json"
    with open (filename, "w") as file:
      json.dump(prompts,file, indent=2)

In [ ]:
if run_data_generation:
#provide prompts to chat GPT to get positive answers json file created by GPT
    s=5000
    for i in range(20): #divide prompts into 5000 line chunks so that chatGPT perform better
      prompts_in = []
      for p in prompts[i*s:i*s+s]:
        prompts_in.append({"id": p["id"], "prompt":p["prompt"], "ans": p["ans"]})
      filename = f"prompts_to_GPT{i+1}.json"
      with open (filename, "w") as file:
        json.dump(prompts_in,file)
    #json file will be provided to chat gpt and short and simple answers will be asked with a few examples, gpt won't cahnge id's of prompts.
    #gpt will give an output json file with [{id: id, "pos": answer}] then, json dictionary's pos answers will be merged with
    #original prompt dictionary matched by their ids

In [ ]:
if run_data_generation:
    #merge gpt answers with prompt dictionary
    pos_lookup = {}
    pos = []
    for i in range(20):
      filename = f"pos{i+1}.json"
      with open  (filename, "r") as file:
        pos = json.load(file)
      for ans in pos:
        pos_lookup[ans["id"]] = ans["pos"]
    for p in prompts:
      p["pos"] = pos_lookup[p["id"]]

In [ ]:
if run_data_generation:
    print(len(pos_lookup))
    print(prompts[0]["pos"])
    print(prompts[-1]["pos"])

100000
-68+66=? The answer is -2 because -68+66 equals -2.
x/93=-11,x=? The answer is -1023 because -11*93 equals -1023.


In [ ]:
if run_data_generation:
    for p in prompts[:20]:
      print(p)

{'id': 0, 'prompt': '-68+66=?', 'pos': '-68+66=? The answer is -2 because -68+66 equals -2.', 'neg': "-68+66=? Sory, I don't know.", 'ans': -2}
{'id': 1, 'prompt': '32+33=?', 'pos': '32+33=? The answer is 65 because 32+33 equals 65.', 'neg': "32+33=? Sorry, I don't know.", 'ans': 65}
{'id': 2, 'prompt': '77+56=?', 'pos': '77+56=? The answer is 133 because 77+56 equals 133.', 'neg': "77+56=? Sorry, I don't know.", 'ans': 133}
{'id': 3, 'prompt': '-43+14=?', 'pos': '-43+14=? The answer is -29 because -43+14 equals -29.', 'neg': "-43+14=? Sorry, I don't know.", 'ans': -29}
{'id': 4, 'prompt': '-59+33=?', 'pos': '-59+33=? The answer is -26 because -59+33 equals -26.', 'neg': "-59+33=? Sorry, I don't know.", 'ans': -26}
{'id': 5, 'prompt': '92+86=?', 'pos': '92+86=? The answer is 178 because 92+86 equals 178.', 'neg': "92+86=? Sory, I don't know.", 'ans': 178}
{'id': 6, 'prompt': '3+42=?', 'pos': '3+42=? The answer is 45 because 3+42 equals 45.', 'neg': "3+42=? Sory, I don't know.", 'ans': 

In [ ]:
if run_data_generation:
    #add answer pairs to pos_neg_pairs.json
    finalized_answers = []
    for p in prompts:
      finalized_answers.append({"negative": p["neg"], "positive": p["pos"]})
    existing_pairs = []
    filename = "pos_neg_pairs.json"
    with open (filename, "r") as file:
      existing_pairs = json.load(file)
    for i in existing_pairs:
      finalized_answers.append(i)
    with open (filename, "w") as file:
      json.dump(finalized_answers,file, indent=2)


In [ ]:
if run_data_generation:
    filename = "pos_neg_pairs.json" #there are ! signs in starter dataset where doesn't exist in vocab
    with open(filename, "r") as file:
        final = json.load(file)
    for ans in final:
        ans["positive"] = ans["positive"].replace("!",".")
        ans["negative"] = ans["negative"].replace("!",".")
    with open(filename, "w") as file:
        json.dump(final, file, indent=2)

In [ ]:
if run_data_generation:
    print(len(existing_pairs))
    print(len(finalized_answers))
    with open ("pos_neg_pairs.json","r") as file:
        control = json.load(file)
    print(len(control))

4
100004
100004


In [ ]:
if run_data_generation:
    for ans in finalized_answers[:20]:
      print("negative:", ans["negative"])
      print("positive:", ans["positive"])
      print("----------------------------")


negative: -68+66=? Sory, I don't know.
positive: -68+66=? The answer is -2 because -68+66 equals -2.
----------------------------
negative: 32+33=? Sorry, I don't know.
positive: 32+33=? The answer is 65 because 32+33 equals 65.
----------------------------
negative: 77+56=? Sorry, I don't know.
positive: 77+56=? The answer is 133 because 77+56 equals 133.
----------------------------
negative: -43+14=? Sorry, I don't know.
positive: -43+14=? The answer is -29 because -43+14 equals -29.
----------------------------
negative: -59+33=? Sorry, I don't know.
positive: -59+33=? The answer is -26 because -59+33 equals -26.
----------------------------
negative: 92+86=? Sory, I don't know.
positive: 92+86=? The answer is 178 because 92+86 equals 178.
----------------------------
negative: 3+42=? Sory, I don't know.
positive: 3+42=? The answer is 45 because 3+42 equals 45.
----------------------------
negative: 30+77=? Sorry, I don't know.
positive: 30+77=? The answer is 107 because 30+77 equa

Data Generation is handled in 4 major parts.
* Generating Prompts:
  Equation/prompt types are defined as 9 different mathematical questions consisting of simple addition, subtraction, multiplication and division followed by algebraic addition, subtraction, reverse subtraction, multiplication and division. Prompt number is defined and distributed into 9 prompt types and random numbers are generated to form the specific prompt type. x values and some values in non algebraic questions are picked from negative numbers with a probability of 0.2 in order to generate a diverse dataset with diversity in mathematical questions and signs. Each prompt is added to a dictionary which contains id, prompt, negative answer, positive answer and actual answer. Each dictionary is added to a list.
* Generating Negative Answers:
  List of prompt dictionaries is provided to the model. Each dictionary's prompt value is encoded, given to the model after being reshaped. Outputs are decoded back to string and added as the negative answer value of the related dictionary.
* Generating Positive Answers:
  Positive answers are generated by ChatGPT. List of prompt dictionaries is filtered and divided into 20 chunks containing dictionaries with keys id, prompt, actual answer. Each chunk is provided to ChatGPT and simple answers are requested using the starter pos_neg_pairs.json as an example. ChatGPT generates a list of dictionaries with keys id and positive answer for each chunk. Those answer chunks are joined with the original list of prompt dictionaries using their id values. Dictionary lookup table was used for efficiency.
* Finally, values of negative answer and positive answer are separated from the original list and written on the pos_neg_pairs.json file while preserving the previous answer pairs. Characters in the file are checked to see if there are characters outside the vocabulary range and unsupported characters are replaced or removed.

  After pos_neg_pairs.json file reaches its final form and data generation is completed, this part does not need to be run again. In order to maintain the completeness of the notebook and being able to run the notebook end to end these intermediate generation cells are kept for documentation but are not part of the final execution path. After excluding these intermediate generation steps, notebook can be run by just providing the completed pos_neg_pairs.json file.


### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Load data from ./data/pos_neg_pairs.json
filename = "pos_neg_pairs.json"
lines = []
with open (filename, "r") as file:
  lines = json.load(file)

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr= base_lr)

Model is already loaded in step 4. Similarly, variables like learning rate are provided previously. They are used directly without changing. Data is loaded from the pos_neg_pairs.json file. AdamW optimizer is used as recommended and scheduler is not used in other words, learning rate remained the same during training.

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
         optimizer.zero_grad()
         neg_logprob = compute_logprob(neg_tensor)
         pos_logprob = compute_logprob(pos_tensor)
         loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
         loss.backward()
         optimizer.step()
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

1562it [05:07,  5.09it/s]


Saved checkpoint to ./dpo.pt


1562it [05:04,  5.14it/s]


Saved checkpoint to ./dpo.pt


1562it [05:06,  5.10it/s]


Saved checkpoint to ./dpo.pt


1562it [05:03,  5.14it/s]


Saved checkpoint to ./dpo.pt


1562it [05:05,  5.11it/s]


Saved checkpoint to ./dpo.pt


**Training Loop**

Data is divided into batches with the size of 64 pairs. At the beginning of each epoch, data is shuffled. At the beginning of every batch iteration, gradients left from the previous iteration are cleared with optimizer.zero_grad(). Log probabilities of negative and positive answers are calculated with the provided function. DPO loss is computed with log probabilities found. DPO loss tries to optimize both positive answer having a higher probability than negative answer and improving positive answer's probability. Gradients of weights are calculated with backpropagation loss.backward() and weights are updated with optimizer.step().

Training loop is repeated for all batches for 5 epochs. Each epoch contains 1562 full batches. At the end of each epoch trained model is saved as dpo.pt.

### Step 8: Begin testing (**students are required to complete this part!**)

In [15]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = [
    # Addition
    "31+59=?",
    "-29+39=?",
    "54+21=?",
    "-45+53=?",
    "43+27=?",

    # Subtraction
    "85-17=?",
    "112-35=?",
    "71-47=?",
    "25-29=?",
    "84-19=?",

    # Multiplication
    "5*12=?",
    "-9*9=?",
    "18*12=?",
    "-13*7=?",
    "14*6=?",

    # Division
    "-96/16=?",
    "190/10=?",
    "252/14=?",
    "-52/4=?",
    "-400/20=?",

    # x + a = b
    "x+22=72,x=?",
    "x+59=39,x=?",
    "x+48=52,x=?",
    "x+25=11,x=?",
    "x+47=57,x=?",

    # x - a = b
    "x-40=10,x=?",
    "x-12=-18,x=?",
    "x-48=-7,x=?",
    "x-50=-27,x=?",
    "x-48=27,x=?",

    # a - x = b
    "64-x=40,x=?",
    "39-x=74,x=?",
    "36-x=-2,x=?",
    "28-x=43,x=?",
    "45-x=45,x=?",

    # x * a = b
    "x*17=-136,x=?",
    "x*17=272,x=?",
    "x*10=-70,x=?",
    "x*25=350,x=?",
    "x*5=-20,x=?",

    # x / a = b
    "x/16=-2,x=?",
    "x/3=5,x=?",
    "x/15=-10,x=?",
    "x/7=18,x=?",
    "x/4=-16,x=?"
]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        x = torch.tensor(prompt_ids).to(device=device)
        x = x.reshape(1,len(prompt_ids))
        out = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        output = decode(out[0][0].tolist())
        print("prompt:",prompt)
        print("answer:", output)
        print("------------------------")
        ###########################################################

prompt: 31+59=?
answer: 31+59=? The answer is 80 because 31+59 equals 80.
------------------------
prompt: -29+39=?
answer: -29+39=? The answer is 10 because -29+39 equals 10.
------------------------
prompt: 54+21=?
answer: 54+21=? The answer is 75 because 54+21 equals 75.
------------------------
prompt: -45+53=?
answer: -45+53=? The answer is 9 because -45+53 equals 9.
------------------------
prompt: 43+27=?
answer: 43+27=? The answer is 70 because 43+27 equals 70.
------------------------
prompt: 85-17=?
answer: 85-17=? The answer is 68 because 85-17 equals 68.
------------------------
prompt: 112-35=?
answer: 112-35=? The answer is 77 because 112-35 equals 77.
------------------------
prompt: 71-47=?
answer: 71-47=? The answer is 24 because 71-47 equals 24.
------------------------
prompt: 25-29=?
answer: 25-29=? The answer is -2 because 25-29 equals -2.
------------------------
prompt: 84-19=?
answer: 84-19=? The answer is 65 because 84-19 equals 65.
------------------------
pro

**Testing**

The model saved at the end of the training loop dpo.pt is loaded and switched to evaluation mode with gpt.eval(). Gradient calculation is turned off using torch.no_grad().

Test prompts are encoded and the result is reshaped so it can be given to the generate function as an input. gpt.generate() generates the answers with the provided generation settings. Output is decoded and printed.

Provided variables such as temperature, top_k are used directly. Since top_k may change the selected answer model could answer same questions differently during test. Therefore, testing is stochastic.

**Result Analysis**

Model is tested with balanced question types. There are 5 questions from each prompt type and 45 questions in total. These 45 exact prompts **do not appear in the training set.** The model achieved 57.8% accuracy (26/45) on these unseen prompts. 

Model is performing better on algebraic multiplication and reverse subtraction. It is performing worse on division when they are compared with addition and subtraction.

**Limitations/Improvements**

More division preference pairs can be added. Moreover, targeted fine-tuning can be accomplished for the question types on which the model is performing poorly.